# Tratamento dos Dados — Municípios e Regionais de Saúde

---
## 1. Imports

In [1]:
import unicodedata
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

---
## 2. Configurações Gerais


In [2]:
# ── Caminhos de entrada ───────────────────────────────────────────────────────
ENTRADA_MUNICIPIOS  = Path("/home/anna/tcc/Limpeza e Integração/Municípios/Relacao-de-municipios-por-vinculacao-administrativa-090.xlsx")
ENTRADA_MORTALIDADE = Path("/home/anna/tcc/Seleção e Transformação/Mortalidade/tabela_mortes_por_faixa_etária.csv")

# ── Caminho de saída ──────────────────────────────────────────────────────────
SAIDA = Path("/home/anna/tcc/Seleção e Transformação/Municípios/tabela_municipios_regionais.csv")

# ── Parâmetros de leitura do Excel ────────────────────────────────────────────
ABA_EXCEL  = "RegionalizaçãoMG2024"   # nome da aba da planilha SES-MG
HEADER_ROW = 3                         # linha do cabeçalho (0-indexado)

# ── Nomes das colunas no CSV de mortalidade ───────────────────────────────────
COL_MUNICIPIO_MORT = "MUNICÍPIO"
COL_REGIONAL_MORT  = "REGIONAL DE SAÚDE"

# ── Correções de grafia: nome_no_excel → nome_na_mortalidade ─────────────────
# Adicione aqui qualquer município com grafia divergente entre as duas fontes.
CORRECOES_NOMES = {
    "brazopolis"           : "brasopolis",
    "dona euzebia"         : "dona eusebia",
    "sao tome das letras"  : "sao thome das letras",
}

# ── Regionais preenchidas manualmente ─────────────────────────────────────────
# Municípios sem nenhum registro na base de mortalidade (portanto sem
# REGIONAL DE SAÚDE para herdar no join) — nunca tiveram óbito por
# câncer de mama registrado em 1979-2024. Regional informada manualmente
# por código IBGE, conferida uma a uma.
REGIONAIS_MANUAIS = {
    3106655: "Montes Claros",       # Berizal
    3110202: "Ponte Nova",          # Cajuri
    3116407: "Passos",              # Claraval
    3118502: "Pouso Alegre",        # Consolação
    3122355: "Pedra Azul",          # Divisa Alegre
    3123007: "São João Del Rei",    # Dores de Campos
    3123502: "Uberlândia",          # Douradoquara
    3127909: "Uberlândia",          # Grupiara
    3143153: "Pedra Azul",          # Monte Formoso
    3147808: "Juiz de Fora",        # Passa Vinte
    3160454: "Montes Claros",       # Santo Antônio do Retiro
    3168200: "Divinópolis",         # Tapiraí
    3169109: "Pouso Alegre",        # Toledo
}

SAIDA.parent.mkdir(parents=True, exist_ok=True)
print(f"✅ Pasta de saída: {SAIDA.parent}")

✅ Pasta de saída: /home/anna/tcc/Seleção e Transformação/Municípios


---
## 3. Funções Auxiliares

`normalizar_texto` é a única função utilitária necessária neste pipeline —
remove acentos e converte para minúsculas para permitir o join entre as duas fontes.

In [3]:
def normalizar_texto(texto) -> str:
    """Remove acentos e converte para minúsculas. Trata NaN como string vazia."""
    texto = "" if pd.isna(texto) else str(texto).strip()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return texto.lower()



---
## 4. Carregamento dos Dados

### 4.1 Municípios — planilha SES-MG

Lê a aba `RegionalizaçãoMG2024` do Excel, extraindo código IBGE e nome do município.

In [4]:
df_excel = pd.read_excel(ENTRADA_MUNICIPIOS, sheet_name=ABA_EXCEL, header=HEADER_ROW)

tabela = pd.DataFrame({
    "codigo_ibge"           : df_excel["Código IBGE"],
    "municipio_original"    : df_excel["Município "].astype(str).str.strip(),
    "municipio_normalizado" : df_excel["Município "].map(normalizar_texto),
})

# Aplica correções de grafia antes do merge
tabela["municipio_normalizado"] = tabela["municipio_normalizado"].replace(CORRECOES_NOMES)

print(f"✅ Excel carregado: {len(tabela)} municípios")
display(tabela.head(3))

✅ Excel carregado: 853 municípios


,codigo_ibge,municipio_original,municipio_normalizado
0,3100104,Abadia dos Dourados,abadia dos dourados
1,3100203,Abaeté,abaete
2,3100302,Abre Campo,abre campo


### 4.2 Regionais de Saúde — CSV de mortalidade

In [5]:
mort = pd.read_csv(ENTRADA_MORTALIDADE, encoding="utf-8-sig")

regionais = (
    mort[[COL_MUNICIPIO_MORT, COL_REGIONAL_MORT]]
    .drop_duplicates(subset=COL_MUNICIPIO_MORT)
    .assign(municipio_normalizado=lambda d: d[COL_MUNICIPIO_MORT].map(normalizar_texto))
    .rename(columns={COL_REGIONAL_MORT: "regional_original"})
    [["municipio_normalizado", "regional_original"]]
)

print(f"✅ CSV de mortalidade carregado: {len(regionais)} municípios únicos com regional")
display(regionais.head(3))

✅ CSV de mortalidade carregado: 853 municípios únicos com regional


,municipio_normalizado,regional_original
0,alfenas,Alfenas
1,alterosa,Alfenas
2,arceburgo,Alfenas


---
## 5. Análise Inicial

Verificação rápida das duas fontes antes do join: dimensões e valores ausentes.

In [6]:
display(Markdown(f"**Municípios no Excel:** {len(tabela)} | **Municípios com regional:** {len(regionais)}"))

# Municípios no Excel que não estão no CSV de mortalidade (serão NaN após o join)
chaves_excel = set(tabela["municipio_normalizado"].dropna())
chaves_mort  = set(regionais["municipio_normalizado"].dropna())
sem_regional_preview = chaves_excel - chaves_mort

display(Markdown(f"**Municípios sem correspondência no CSV de mortalidade (prévia):** {len(sem_regional_preview)}"))
if sem_regional_preview:
    for m in sorted(sem_regional_preview)[:10]:
        print(f"  - {m}")
    if len(sem_regional_preview) > 10:
        print(f"  ... e mais {len(sem_regional_preview) - 10}")

**Municípios no Excel:** 853 | **Municípios com regional:** 853

**Municípios sem correspondência no CSV de mortalidade (prévia):** 1

  - passa vinte


---
## 6. Join pelo Nome Normalizado

Une a tabela de municípios com as regionais de saúde usando `municipio_normalizado` como chave.
Adiciona também `regional_normalizada` para facilitar joins futuros.

In [7]:
tabela = tabela.merge(regionais, on="municipio_normalizado", how="left")
tabela["regional_normalizada"] = tabela["regional_original"].map(normalizar_texto)

print(f"✅ Join concluído. Shape: {tabela.shape}")
display(tabela.head(5))

✅ Join concluído. Shape: (853, 5)


,codigo_ibge,municipio_original,municipio_normalizado,regional_original,regional_normalizada
0,3100104,Abadia dos Dourados,abadia dos dourados,Uberlândia,uberlandia
1,3100203,Abaeté,abaete,Sete Lagoas,sete lagoas
2,3100302,Abre Campo,abre campo,Manhumirim,manhumirim
3,3100401,Acaiaca,acaiaca,Ponte Nova,ponte nova
4,3100500,Açucena,acucena,Coronel Fabriciano,coronel fabriciano


---
## 6b. Preenchimento Manual — Municípios sem Registro na Mortalidade

Aplica `REGIONAIS_MANUAIS` (Célula 2) apenas às linhas em que o join
acima não encontrou regional — municípios sem nenhum óbito por câncer
de mama registrado na base de mortalidade, portanto sem
`REGIONAL DE SAÚDE` para herdar.

In [8]:
linhas_sem_regional = tabela["codigo_ibge"].isin(REGIONAIS_MANUAIS) & tabela["regional_original"].isna()

tabela.loc[linhas_sem_regional, "regional_original"] = tabela.loc[linhas_sem_regional, "codigo_ibge"].map(REGIONAIS_MANUAIS)
tabela["regional_normalizada"] = tabela["regional_original"].map(normalizar_texto)

print(f"✅ {linhas_sem_regional.sum()} município(s) preenchido(s) manualmente via REGIONAIS_MANUAIS.")
display(tabela[linhas_sem_regional][["codigo_ibge", "municipio_original", "regional_original"]].reset_index(drop=True))

✅ 1 município(s) preenchido(s) manualmente via REGIONAIS_MANUAIS.


,codigo_ibge,municipio_original,regional_original
0,3147808,Passa Vinte,Juiz de Fora


---
## 7. Validação Pós-Join

Verifica municípios sem regional associada e exibe um relatório completo.

In [9]:
sem_regional = tabela["regional_original"].isna().sum()
com_regional = tabela["regional_original"].notna().sum()

print("╔══════════════════════════════════════════════════╗")
print("║              Resultado do join                   ║")
print("╠══════════════════════════════════════════════════╣")
print(f"║  Total de municípios        : {len(tabela):<18} ║")
print(f"║  Com regional identificada  : {com_regional:<18} ║")
print(f"║  Sem regional (NaN)         : {sem_regional:<18} ║")
print("╚══════════════════════════════════════════════════╝")

if sem_regional:
    display(Markdown(f"⚠️ **{sem_regional} município(s) sem regional — ausentes na base de mortalidade:**"))
    display(
        tabela[tabela["regional_original"].isna()][["codigo_ibge", "municipio_original"]]
        .reset_index(drop=True)
        .style.hide(axis="index")
    )
else:
    display(Markdown("✅ **Todos os municípios tiveram regional identificada.**"))

╔══════════════════════════════════════════════════╗
║              Resultado do join                   ║
╠══════════════════════════════════════════════════╣
║  Total de municípios        : 853                ║
║  Com regional identificada  : 853                ║
║  Sem regional (NaN)         : 0                  ║
╚══════════════════════════════════════════════════╝


✅ **Todos os municípios tiveram regional identificada.**

---
## 8. Salvando o Resultado

In [10]:
# Padroniza os nomes das colunas em CAIXA ALTA
tabela = tabela.rename(columns={
    "codigo_ibge"           : "CÓDIGO IBGE",
    "municipio_original"    : "MUNICÍPIO ORIGINAL",
    "municipio_normalizado" : "MUNICÍPIO NORMALIZADO",
    "regional_original"     : "REGIONAL ORIGINAL",
    "regional_normalizada"  : "REGIONAL NORMALIZADA",
})

tabela.to_csv(SAIDA, index=False, encoding="utf-8-sig")

print("╔══════════════════════════════════════════════════════╗")
print("║              Arquivo gerado com sucesso              ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  Arquivo : {str(SAIDA.name):<42}║")
print(f"║  Linhas  : {len(tabela):<42}║")
print(f"║  Colunas : {list(tabela.columns)}")
print("╚══════════════════════════════════════════════════════╝")

display(Markdown("**Amostra da tabela final:**"))
display(tabela.head(5))

╔══════════════════════════════════════════════════════╗
║              Arquivo gerado com sucesso              ║
╠══════════════════════════════════════════════════════╣
║  Arquivo : tabela_municipios_regionais.csv           ║
║  Linhas  : 853                                       ║
║  Colunas : ['CÓDIGO IBGE', 'MUNICÍPIO ORIGINAL', 'MUNICÍPIO NORMALIZADO', 'REGIONAL ORIGINAL', 'REGIONAL NORMALIZADA']
╚══════════════════════════════════════════════════════╝


**Amostra da tabela final:**

,CÓDIGO IBGE,MUNICÍPIO ORIGINAL,MUNICÍPIO NORMALIZADO,REGIONAL ORIGINAL,REGIONAL NORMALIZADA
0,3100104,Abadia dos Dourados,abadia dos dourados,Uberlândia,uberlandia
1,3100203,Abaeté,abaete,Sete Lagoas,sete lagoas
2,3100302,Abre Campo,abre campo,Manhumirim,manhumirim
3,3100401,Acaiaca,acaiaca,Ponte Nova,ponte nova
4,3100500,Açucena,acucena,Coronel Fabriciano,coronel fabriciano
